# RTSA End-to-End Walkthrough

This notebook runs the full RTSA pipeline on real GSM8K CoT traces:

1. Load traces and extract typed reasoning graphs
2. Validate against NGS structural rules (failure-mode taxonomy)
3. Detect redundancy and prune (with savings uncertainty band)
4. Correlate structure with correctness (synthetic validation)
5. Cluster steps into semantic macro-steps
6. Train the step-level correctness classifier on NGS-rule labels
7. Calibrate pruning thresholds on annotated data

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
from pprint import pprint

from utils.data_loader import load_cot_traces
from extractors.rule_based import RuleBasedExtractor
from core.ngs_validator import NGSValidator, classify_failure_mode
from analysis.prune import RedundancyAnalyzer, PruneConfig
from analysis.step_clustering import StepClusterer
from analysis.step_classifier import StepCorrectnessClassifier, StepFeatureExtractor
from core.robust_tsi import UnsupervisedTSI, bootstrap_tsi_ci

print("RTSA ready")

In [ ]:
traces = load_cot_traces(str(PROJECT_ROOT / "data/raw_cots/gsm8k_50.jsonl"))[:10]
print(f"Loaded {len(traces)} GSM8K traces")
print("Example question:", traces[0]["question"][:120])

In [ ]:
extractor = RuleBasedExtractor()
graphs = []
for i, t in enumerate(traces):
    g = extractor.extract(
        t.get("cot_text", ""),
        trace_id=t.get("question_id", f"t{i}"),
        answer=t.get("answer", ""),
    )
    graphs.append(g)

g0 = graphs[0]
print(f"Trace '{g0.trace_id}': {len(g0.nodes)} nodes, {len(g0.edges)} edges")
pprint([(n.id, n.type.value, n.text[:50]) for n in g0.nodes[:6]])

In [ ]:
validator = NGSValidator()
all_violations = 0
for g in graphs:
    valid, violations = validator.validate(g)
    all_violations += len(violations)

_, v0 = validator.validate(g0)
modes = classify_failure_mode(v0)
print(f"Total NGS violations across {len(graphs)} traces: {all_violations}")
print(f"Trace '{g0.trace_id}' failure modes:")
for mode, viols in modes.items():
    print(f"  {mode}: {len(viols)} violation(s)")

In [ ]:
analyzer = RedundancyAnalyzer(config=PruneConfig())
report = analyzer.analyze(g0, apply_pruning=True)

lo, hi = report.savings_range()
print(f"PruningReport for {report.trace_id}")
print(f"  Nodes: {report.original_n_nodes} -> "
      f"{len(report.pruned_graph.nodes) if report.pruned_graph else 'N/A'}")
print(f"  Est. token savings: {report.total_estimated_savings} (range {lo}-{hi})")
print(f"  Integrity score: {report.structural_integrity_score:.2f}")
for r in report.redundancy_regions:
    print(f"  [{r.region_type}] conf={r.confidence:.2f} action={r.suggested_action}")

In [ ]:
# Structure <-> correctness correlation, validated on synthetic data
# (labels vary by construction; GSM8K human solutions are all correct)
from experiments.correlation_analysis import synthetic_graphs, run as corr_run

syn = synthetic_graphs(n=60, seed=42)
labels = {g.trace_id: g.metadata.get("correct", True) for g in syn}
report = corr_run(syn, labels)
print(f"{report['n_graphs']} graphs, {report['n_incorrect']} incorrect")
for c in report["correlations"][:5]:
    sig = "*" if c["significant"] else " "
    print(f"  {c['metric']:<18} rho={c['spearman_rho']:+.3f} p={c['p_value']:.4f} {sig}")

In [ ]:
# Semantic step clustering: merge chain segments into macro-steps
clusterer = StepClusterer(method="heuristic", min_similarity=0.30)
merged = clusterer.cluster(g0)

print(f"Nodes before: {len(g0.nodes)} -> after clustering: {len(merged.nodes)}")
merged_types = [n.type.value for n in merged.nodes]
print(f"Merged graph node types: {merged_types}")
info = merged.metadata.get("step_clustering", {})
print(f"Merged segments: {info.get('n_merged', 0)}")

In [ ]:
# Train the step-level correctness classifier on NGS-rule labels
from experiments.annotate_steps import annotate_trace

validator = NGSValidator()
labels = []
for g in graphs:
    recs = annotate_trace(g, validator)
    labels.append({r["node_id"]: r["is_correct"] for r in recs})

clf = StepCorrectnessClassifier().fit(graphs, labels)
n_error = sum(1 for lab in labels for v in lab.values() if not v)
print(f"Trained on {sum(len(l) for l in labels)} labelled nodes ({n_error} errors)")

import numpy as np
imps = clf.feature_importance()
top = sorted(imps.items(), key=lambda kv: -kv[1])[:6]
print("Top structural features:")
for name, imp in top:
    print(f"  {name:<24} {imp:.4f}")

diag = clf.analyze(g0, threshold=0.5)
print(f"Diagnosed {len(diag)} steps; flagged errors: "
      f"{[d.node_id for d in diag if d.is_error]}")

In [ ]:
# Threshold calibration on synthetic annotations
from experiments.calibrate_thresholds import (
    synthetic_annotated_graphs, run as cal_run,
)

cal_graphs, cal_annotations = synthetic_annotated_graphs(n=40, seed=42)
cal_report = cal_run(cal_graphs, cal_annotations, iterations=1, metric="f1")
print(f"Best thresholds after 1 descent round:")
for k, v in cal_report["best_params"].items():
    print(f"  {k}: {v}")
agg = cal_report["best_aggregate"]
print(f"Aggregate P/R/F1: {agg['precision']:.3f} / "
      f"{agg['recall']:.3f} / {agg['f1']:.3f}")

In [ ]:
# Unsupervised graph similarity + bootstrap CI
usi = UnsupervisedTSI()
s = usi.similarity(graphs[0], graphs[1])
mean, lo, hi = bootstrap_tsi_ci(usi.similarity, graphs[0], graphs[1], n_bootstrap=200)
print(f"Similarity(t0, t1) = {s:.3f}")
print(f"Bootstrap CI: mean={mean:.3f}, 95% CI=[{lo:.3f}, {hi:.3f}]")

## Summary

The notebook demonstrates the complete RTSA workflow on real traces:
extraction -> NGS validation with failure modes -> redundancy pruning with
uncertainty bands -> structure/correctness correlation -> semantic step
clustering -> learned step-level diagnosis -> threshold calibration.

For production runs use the versioned entrypoint instead:
`python -m experiments.run all --dataset gsm8k`.